# M3_F06 — CARRIER
## *Le Transporteur de Croisière — Mode III Ascension*

**Mission** : Interpoler frames PNG (RIFE 24→60fps) + encoder MP4 H.264 + mux audio + overlay texte

**Entrée** : `EXODUS_V3/M3/F05_ALCHEMIST/OUT_FRAMES/frame_*.png`  
**Sortie** : `EXODUS_V3/M3/F06_CARRIER/OUT/FINAL_OUTPUT.mp4`

---
> *Ad Majorem Gloriam Imperatoris*

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 1 — MONTAGE DRIVE + INSTALLATION
# ═══════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

# Vérification GPU T4
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU :', result.stdout.strip() or 'CPU mode')

# Installation dépendances
!pip install flask flask-cors -q

# Copier les scripts dans /content
import shutil
from pathlib import Path

SCRIPTS_DRIVE = Path('/content/drive/MyDrive/EXODUS_V3/M3/F06_CARRIER/CODEBASE')
for f in ['m3_f06_flask.py', 'm3_f06_pipeline.py', 'm3_f06_monitor.html']:
    src = SCRIPTS_DRIVE / f
    if src.exists():
        shutil.copy(src, f'/content/{f}')
        print(f'Copié : {f}')
    else:
        print(f'MANQUANT : {f} — vérifier Drive')

print('\n[OK] Cellule 1 terminée')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 2 — VÉRIFICATION INPUTS
# ═══════════════════════════════════════════════════════════════
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/EXODUS_V3/M3')
IN_FRAMES  = DRIVE_ROOT / 'F05_ALCHEMIST' / 'OUT_FRAMES'
AUDIO_PATH = DRIVE_ROOT / 'SHARED' / 'audio.mp3'
F01_REPORT = DRIVE_ROOT / 'F01_VALIDATION' / 'OUT_REPORT' / 'm3_f01_report.json'

frames = sorted(IN_FRAMES.glob('frame_*.png')) if IN_FRAMES.exists() else []
print(f'Frames détectées : {len(frames)}')
if frames:
    print(f'  Première : {frames[0].name}')
    print(f'  Dernière : {frames[-1].name}')

print(f'Audio     : {"OUI" if AUDIO_PATH.exists() else "NON"}')
print(f'F01 report: {"OUI" if F01_REPORT.exists() else "NON"}')

if len(frames) == 0:
    print('\n[ATTENTION] Aucune frame — F05 doit être exécuté en premier')
else:
    duration = len(frames) / 24
    print(f'\nDurée estimée   : {duration:.1f}s @ 24fps')
    print(f'Frames 60fps    : ~{int(duration * 60)} frames après RIFE')
    print(f'Taille estimée  : ~{len(frames) * 3 / 1024:.1f} GB')
    print('\n[OK] Cellule 2 terminée — Prêt pour encodage')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 3 — LANCER LE SERVEUR FLASK
# ═══════════════════════════════════════════════════════════════
import sys
sys.path.insert(0, '/content')

import threading
from m3_f06_flask import start_server

PORT = 8080
t = threading.Thread(target=start_server, args=(PORT,), daemon=True)
t.start()

import time; time.sleep(2)

# Exposer le port Colab
from google.colab.output import serve_kernel_port_as_window
serve_kernel_port_as_window(PORT)

print(f'[OK] Serveur Flask actif sur port {PORT}')
print('     Ouvrez la fenêtre pop-up pour accéder au CARRIER')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 4 — OPTIONNEL : ENCODAGE DIRECT PYTHON (sans viewer)
# Pour usage headless / automatisé
# ═══════════════════════════════════════════════════════════════
# Décommenter et configurer si besoin d'un run sans navigateur

# from m3_f06_pipeline import run_rife, run_ffmpeg_encode, run_audio_mux, run_overlay, cleanup
# from pathlib import Path
#
# DRIVE_ROOT  = Path('/content/drive/MyDrive/EXODUS_V3/M3')
# IN_FRAMES   = DRIVE_ROOT / 'F05_ALCHEMIST' / 'OUT_FRAMES'
# RIFE_OUT    = DRIVE_ROOT / 'F06_CARRIER' / 'RIFE_FRAMES'
# OUT_DIR     = DRIVE_ROOT / 'F06_CARRIER' / 'OUT'
# AUDIO_PATH  = DRIVE_ROOT / 'SHARED' / 'audio.mp3'
# FPS_TARGET  = 60
#
# OUT_DIR.mkdir(parents=True, exist_ok=True)
# RIFE_OUT.mkdir(parents=True, exist_ok=True)
#
# print('RIFE...')
# run_rife(IN_FRAMES, RIFE_OUT, FPS_TARGET, 24,
#          progress_cb=lambda p, m: print(f'  {p:.0f}% {m}'))
#
# print('Encodage H.264...')
# tmp_novid = OUT_DIR / 'tmp_novid.mp4'
# run_ffmpeg_encode(RIFE_OUT, tmp_novid, FPS_TARGET,
#                   progress_cb=lambda p, m: print(f'  {p:.0f}% {m}'))
#
# print('Audio mux...')
# tmp_audio = OUT_DIR / 'tmp_audio.mp4'
# if AUDIO_PATH.exists():
#     run_audio_mux(tmp_novid, AUDIO_PATH, tmp_audio)
# else:
#     import shutil; shutil.copy(tmp_novid, tmp_audio)
#
# import shutil
# shutil.copy(tmp_audio, OUT_DIR / 'FINAL_OUTPUT.mp4')
# cleanup(IN_FRAMES, RIFE_OUT, [tmp_novid, tmp_audio])
# print('[OK] FINAL_OUTPUT.mp4 sauvé sur Drive')

print('Cellule 4 : commentée par défaut — décommenter pour run headless')